In [1]:
# =============================================================================
# 1. IMPORTS & SETUP
# =============================================================================
import pandas as pd
import numpy as np
import os
import faiss
import joblib
from sentence_transformers import SentenceTransformer

# Update this path to your exact project folder
base_path = r"C:\Bhumala\Real Estate build 28-apr-2026\Real_Estate_Platform 24-04-26\Real_Estate_Platform\Main_App\ml_models"
excel_file = os.path.join(base_path, "final_cleaned_datatset.xlsx")

# Output paths
csv_output = os.path.join(base_path, "df.csv")
faiss_output = os.path.join(base_path, "faiss_index.bin")
pkl_output = os.path.join(base_path, "cities.pkl")

# =============================================================================
# 2. LOAD DATA AND FIX THE DB_ID BRIDGE
# =============================================================================
if not os.path.exists(excel_file):
    print(f"❌ Error: File not found at {excel_file}")
else:
    xls = pd.ExcelFile(excel_file)
    all_dfs = []

    for sheet_name in xls.sheet_names:
        df_sheet = pd.read_excel(xls, sheet_name=sheet_name)
        
        # --- CRITICAL FIX: ENSURE DB_ID EXISTS ---
        # If your excel has an 'id' column, we use it. 
        # If not, we use the row index + 1 (Starting at 1 to match Django PKs)
        if 'id' in df_sheet.columns:
            df_sheet['db_id'] = df_sheet['id']
        else:
            df_sheet['db_id'] = df_sheet.index + 1
            print(f"⚠️ Note: '{sheet_name}' had no id column, generated IDs from index.")

        # --- TAG CATEGORIES ---
        df_sheet['source_sheet'] = sheet_name
        
        # --- UNIFY COLUMNS ---
        # Map monthly_rent or rent to expected_price so the AI has one price column to read
        if 'monthly_rent' in df_sheet.columns:
            df_sheet['expected_price'] = df_sheet['monthly_rent']
        elif 'rent' in df_sheet.columns:
            df_sheet['expected_price'] = df_sheet['rent']
        
        # Fill missing text columns
        for col in ['city', 'locality', 'bhk_type']:
            if col not in df_sheet.columns:
                df_sheet[col] = "Not Specified"
            df_sheet[col] = df_sheet[col].fillna("Not Specified").astype(str)

        all_dfs.append(df_sheet)

    # Combine all sheets
    df = pd.concat(all_dfs, ignore_index=True)
    
    # Save the CSV for Django
    df.to_csv(csv_output, index=False)
    print(f"✅ Created df.csv with {len(df)} rows and 'db_id' column.")

    # =========================================================================
    # 3. GENERATE AI BRAIN (FAISS INDEX)
    # =========================================================================
    print("⏳ Creating AI Embeddings... this may take a moment.")
    
    # Create the text description the AI uses to "read" the property
    df['combined_text'] = (
        df['source_sheet'] + " in " + 
        df['locality'] + " " + 
        df['city'] + " " + 
        df['bhk_type'].astype(str)
    ).str.lower()

    # Load Model
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(df['combined_text'].tolist(), show_progress_bar=True)

    # Create and Save FAISS
    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(np.array(embeddings).astype('float32'))
    faiss.write_index(index, faiss_output)
    print(f"✅ FAISS Index saved to: {faiss_output}")

    # =========================================================================
    # 4. EXPORT CITIES
    # =========================================================================
    cities = sorted(df['city'].unique().tolist())
    joblib.dump(cities, pkl_output)
    print(f"✅ Cities pkl saved. Process Complete.")

⚠️ Note: 'PG Data' had no id column, generated IDs from index.
⚠️ Note: 'Commercial Data' had no id column, generated IDs from index.
⚠️ Note: 'Residential Data' had no id column, generated IDs from index.
⚠️ Note: 'Resale Residential' had no id column, generated IDs from index.
⚠️ Note: 'Industrial Resale' had no id column, generated IDs from index.
⚠️ Note: 'Plot Resale' had no id column, generated IDs from index.
⚠️ Note: 'Agricultural Data' had no id column, generated IDs from index.
⚠️ Note: 'Commercial Resale' had no id column, generated IDs from index.
✅ Created df.csv with 32000 rows and 'db_id' column.
⏳ Creating AI Embeddings... this may take a moment.


C:\Users\Hp\AppData\Local\Temp\ipykernel_11032\651199513.py:72: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df['combined_text'] = (


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1000 [00:00<?, ?it/s]

✅ FAISS Index saved to: C:\Bhumala\Real Estate build 28-apr-2026\Real_Estate_Platform 24-04-26\Real_Estate_Platform\Main_App\ml_models\faiss_index.bin
✅ Cities pkl saved. Process Complete.
